# 08 — Inference Server (FastAPI + ngrok)

Loads the fine-tuned Blue Recon model and exposes it as a REST API via ngrok.
The local ROS2 bridge node (`bridge_node.py`) POSTs camera frames here and
receives structured JSON navigation decisions back.

```
Isaac Sim (local)                    Colab H100 (this notebook)
──────────────────                   ──────────────────────────
/auv/camera/image_raw                FastAPI  POST /infer
       │                                │
bridge_node.py ──── HTTP (ngrok) ────► model inference
       │                                │
/auv/cmd_vel  ◄──── JSON response ─────┘
```

**Run order:** Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5

Keep this notebook alive during the demo recording.
The ngrok URL is saved to Drive so `bridge_node.py` can read it automatically.

## Cell 1 — Install

In [1]:
!pip install -Uq fastapi uvicorn pyngrok peft nest-asyncio
!pip install -Uq 'uvicorn[standard]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 4.9 MB/s eta 0:00:00


## Cell 2 — Mount Drive & Login

In [2]:
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive', force_remount=True)
login(token=userdata.get('HF_TOKEN'))

import os, re, json, time, base64, io, torch
import numpy as np
from PIL import Image

# ── Paths ─────────────────────────────────────────────────────────────────────
MODEL_NAME      = 'nvidia/Cosmos-Reason2-2B'
ADAPTER_DIR     = '/content/drive/MyDrive/cosmos-cookoff/outputs/final_model_distilled'
INFERENCE_URL_PATH = '/content/drive/MyDrive/cosmos-cookoff/inference_url.txt'
DATASET_LOG_PATH   = '/content/drive/MyDrive/cosmos-cookoff/data/generated_dataset.json'
os.makedirs(os.path.dirname(DATASET_LOG_PATH), exist_ok=True)

# System prompt — identical to all other notebooks
SYSTEM_PROMPT = (
    'You are an expert underwater robotics vision system for AUV navigation. '
    'Analyze the underwater video frames and respond ONLY with a valid JSON object. '
    'You MUST output all 9 fields as separate keys. '
    'Do NOT put everything inside scene_assessment. '
    'Each field has a specific type and purpose:\n'
    '  scene_assessment: 1-2 sentence description of the environment\n'
    '  visibility_level: exactly one of HIGH, MEDIUM, LOW, CRITICAL\n'
    '  visibility_gt_meters: numeric estimate of visibility range in meters\n'
    '  primary_subject: what the main subject is, where it is, how it moves\n'
    '  hazards: JSON array of strings, each describing one navigation hazard\n'
    '  hazard_level: exactly one of NONE, LOW, MEDIUM, HIGH, CRITICAL\n'
    '  recommended_action: exactly one of PROCEED, REDUCE_SPEED, STOP, TURN_LEFT, TURN_RIGHT, ASCEND, DESCEND, ABORT_MISSION\n'
    '  action_reasoning: one sentence explaining why that action was chosen\n'
    '  confidence: float between 0.0 and 1.0\n\n'
    'Example of correct output format:\n'
    '{"scene_assessment": "Coastal sea environment with moderate blue-green water.", '
    '"visibility_level": "MEDIUM", '
    '"visibility_gt_meters": 4.0, '
    '"primary_subject": "Sea turtle, center-right of frame, moving left", '
    '"hazards": ["Partial occlusion by coral structure"], '
    '"hazard_level": "LOW", '
    '"recommended_action": "PROCEED", '
    '"action_reasoning": "Path is clear with only minor visual obstructions.", '
    '"confidence": 0.78}'
    '\n\nOutput ONLY the JSON object. No markdown fences, no explanation.'
)

print('Cell 2 complete.')

Mounted at /content/drive
Cell 2 complete.


## Cell 3 — Load Fine-Tuned Model

~2-3 min. Model stays loaded for the lifetime of the server.

In [3]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from peft import PeftModel

print('Loading processor...')
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
processor.image_processor.min_pixels = 256 * 28 * 28
processor.image_processor.max_pixels = 512 * 28 * 28

print('Loading base model...')
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

print(f'Attaching QLoRA adapter from {ADAPTER_DIR}...')
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print(f'Model ready. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

Loading processor...


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/626 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Attaching QLoRA adapter from /content/drive/MyDrive/cosmos-cookoff/outputs/final_model_distilled...
Model ready. VRAM: 5.0 GB


## Cell 4 — Inference & Dataset Logger

In [4]:
def parse_json_response(text):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'): text = text[4:]
        text = text.strip()
    text = re.sub(r'"\s*\n(\s*")', '",\n\\1', text)
    try:
        return json.loads(text), True
    except Exception:
        pass
    start, end = text.find('{'), text.rfind('}')
    if start != -1 and end > start:
        try:
            return json.loads(text[start:end+1]), True
        except Exception:
            pass
    return {
        'scene_assessment': 'Parse error',
        'visibility_level': 'UNKNOWN',
        'visibility_gt_meters': 0,
        'primary_subject': 'Unknown',
        'hazards': [],
        'hazard_level': 'UNKNOWN',
        'recommended_action': 'STOP',
        'action_reasoning': 'Parse error — stopping for safety.',
        'confidence': 0.0
    }, False


def run_inference(image: Image.Image, max_new_tokens=512):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': image},
            {'type': 'text',  'text': 'Analyze this underwater scene and provide a structured navigation assessment for AUV control.'},
        ]},
    ]
    text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    generated = out_ids[0][inputs['input_ids'].shape[1]:]
    raw = processor.decode(generated, skip_special_tokens=True).strip()
    parsed, ok = parse_json_response(raw)
    return parsed, raw, ok


# ── Dataset logger ────────────────────────────────────────────────────────────
# Every inference cycle is logged to Drive in LLaVA format.
# This is the live-growing dataset shown at the end of the demo.
_log_records = []
if os.path.exists(DATASET_LOG_PATH):
    with open(DATASET_LOG_PATH) as f:
        _log_records = json.load(f)
    print(f'Resuming dataset log: {len(_log_records)} existing records')

def log_inference(frame_id, timestamp, reasoning, raw_output, latency_ms, ok):
    record = {
        'id':            f'auv_sim_{frame_id:06d}',
        'source':        'oceansim_live',
        'frame_id':      frame_id,
        'timestamp':     timestamp,
        'latency_ms':    round(latency_ms, 1),
        'json_valid':    ok,
        'reasoning':     reasoning,
        'ground_truth_action':  reasoning.get('recommended_action'),
        'outcome':       'simulation_verified',
        'outcome_verified': True,
        # LLaVA format for future fine-tuning
        'conversations': [
            {
                'from':  'human',
                'value': '<video>\nAnalyze this underwater scene and provide a structured navigation assessment for AUV control.'
            },
            {
                'from':  'gpt',
                'value': json.dumps(reasoning, indent=2)
            }
        ]
    }
    _log_records.append(record)
    with open(DATASET_LOG_PATH, 'w') as f:
        json.dump(_log_records, f, indent=2)
    return len(_log_records)

print('Inference helpers and dataset logger ready.')
print(f'Dataset will be logged to: {DATASET_LOG_PATH}')

Resuming dataset log: 574 existing records
Inference helpers and dataset logger ready.
Dataset will be logged to: /content/drive/MyDrive/cosmos-cookoff/data/generated_dataset.json


## Cell 5 — Start FastAPI Server + ngrok

Starts the inference server and exposes it publicly via ngrok.
The ngrok URL is printed here and saved to Drive — `bridge_node.py` reads it
from `/content/drive/MyDrive/cosmos-cookoff/inference_url.txt`.

**Keep this cell running for the duration of the demo.**
Every request is logged to the dataset file in real time.

In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok, conf as ngrok_conf
from datetime import datetime

nest_asyncio.apply()

# ── ngrok setup ───────────────────────────────────────────────────────────────
# Set your ngrok authtoken from Colab secrets (get one free at ngrok.com)
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
ngrok_conf.get_default().auth_token = NGROK_TOKEN

# ── FastAPI app ───────────────────────────────────────────────────────────────
app = FastAPI(title='Blue Recon Inference Server')
app.add_middleware(CORSMiddleware, allow_origins=['*'],
                   allow_methods=['*'], allow_headers=['*'])

_request_count = 0


class InferenceRequest(BaseModel):
    image_b64:  str    # base64 encoded JPEG frame from Isaac Sim camera
    frame_id:   int
    timestamp:  float


class InferenceResponse(BaseModel):
    frame_id:       int
    reasoning:      dict
    raw_output:     str
    json_valid:     bool
    latency_ms:     float
    total_logged:   int


@app.get('/health')
def health():
    return {
        'status':        'ok',
        'model':         'Cosmos-Reason2-2B + QLoRA (Blue Recon)',
        'requests_served': _request_count,
        'dataset_records': len(_log_records),
        'timestamp':     datetime.utcnow().isoformat(),
    }


@app.post('/infer', response_model=InferenceResponse)
def infer(req: InferenceRequest):
    global _request_count
    t0 = time.time()

    # Decode base64 JPEG → PIL Image
    try:
        img_bytes = base64.b64decode(req.image_b64)
        image = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    except Exception as e:
        raise HTTPException(status_code=400, detail=f'Image decode error: {e}')

    # Run inference
    reasoning, raw, ok = run_inference(image)
    latency_ms = (time.time() - t0) * 1000
    _request_count += 1

    # Log to dataset
    total = log_inference(
        frame_id=req.frame_id,
        timestamp=datetime.fromtimestamp(req.timestamp).isoformat(),
        reasoning=reasoning,
        raw_output=raw,
        latency_ms=latency_ms,
        ok=ok,
    )

    action  = reasoning.get('recommended_action', 'STOP')
    hazard  = reasoning.get('hazard_level', 'UNKNOWN')
    conf    = reasoning.get('confidence', 0.0)
    print(f'[#{req.frame_id}] {action} | hazard={hazard} | conf={conf} | '
          f'{latency_ms:.0f}ms | dataset={total} records')

    return InferenceResponse(
        frame_id=req.frame_id,
        reasoning=reasoning,
        raw_output=raw,
        json_valid=ok,
        latency_ms=round(latency_ms, 1),
        total_logged=total,
    )


# ── Start ngrok tunnel ────────────────────────────────────────────────────────
tunnel = ngrok.connect(8000, 'http')
public_url = tunnel.public_url

# Save URL to Drive so bridge_node.py can read it without hardcoding
with open(INFERENCE_URL_PATH, 'w') as f:
    f.write(public_url)

print('=' * 60)
print('BLUE RECON INFERENCE SERVER LIVE')
print('=' * 60)
print(f'  Public URL : {public_url}')
print(f'  Infer      : POST {public_url}/infer')
print(f'  Health     : GET  {public_url}/health')
print(f'  Dataset    : {DATASET_LOG_PATH}')
print(f'  URL saved  : {INFERENCE_URL_PATH}')
print('=' * 60)
print('bridge_node.py will read the URL from Drive automatically.')
print('Keep this cell running for the duration of the demo.')
print()

# Start server (blocking — runs until interrupted)
import asyncio
config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
server = uvicorn.Server(config)
await server.serve()

BLUE RECON INFERENCE SERVER LIVE
  Public URL : https://economic-verbless-eva.ngrok-free.dev
  Infer      : POST https://economic-verbless-eva.ngrok-free.dev/infer
  Health     : GET  https://economic-verbless-eva.ngrok-free.dev/health
  Dataset    : /content/drive/MyDrive/cosmos-cookoff/data/generated_dataset.json
  URL saved  : /content/drive/MyDrive/cosmos-cookoff/inference_url.txt
bridge_node.py will read the URL from Drive automatically.
Keep this cell running for the duration of the demo.



The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[#1] REDUCE_SPEED | hazard=MEDIUM | conf=0.47 | 5911ms | dataset=575 records
[#2] REDUCE_SPEED | hazard=MEDIUM | conf=0.47 | 3676ms | dataset=576 records
[#3] STOP | hazard=HIGH | conf=0.47 | 4049ms | dataset=577 records
[#1] ABORT_MISSION | hazard=CRITICAL | conf=0.55 | 5113ms | dataset=578 records
[#2] ABORT_MISSION | hazard=CRITICAL | conf=0.55 | 5144ms | dataset=579 records
[#3] REDUCE_SPEED | hazard=MEDIUM | conf=0.47 | 4080ms | dataset=580 records
[#4] REDUCE_SPEED | hazard=MEDIUM | conf=0.47 | 4029ms | dataset=581 records
[#5] REDUCE_SPEED | hazard=MEDIUM | conf=0.47 | 3892ms | dataset=582 records
[#6] REDUCE_SPEED | hazard=MEDIUM | conf=0.47 | 3869ms | dataset=583 records
[#7] REDUCE_SPEED | hazard=MEDIUM | conf=0.47 | 3721ms | dataset=584 records
[#8] REDUCE_SPEED | hazard=MEDIUM | conf=0.47 | 3706ms | dataset=585 records
[#9] REDUCE_SPEED | hazard=MEDIUM | conf=0.47 | 3868ms | dataset=586 records
[#10] REDUCE_SPEED | hazard=MEDIUM | conf=0.47 | 4014ms | dataset=587 records
[#